# Práctica 2 · Paso 1 — Exposición del territorio a las amenazas del capítulo 1

**Cambio Climático · Ingeniería en Riesgos de Desastres · ESPAM MFL** · Material CC-IRD-P2-01 · Versión 1.0 · noviembre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-2/RD2_01_exposicion.ipynb)

**Objetivo.** Identificar qué hay expuesto en cada cantón de su provincia (personas, servicios críticos, vías) frente a las amenazas que caracterizó en el capítulo 1, y cuáles cantones están más expuestos a cada amenaza.

**Producto.** Tabla de exposición por cantón y amenaza (puntaje 1–5 con sus indicadores en unidades reales), mapas de exposición y la planilla del inventario de elementos expuestos que el equipo completa sin docente.

**Marco.** Riesgo = amenaza × exposición × vulnerabilidad (IPCC AR6, GTII). Este paso trabaja solo la **exposición**: la presencia de personas, bienes y servicios en lugares que pueden ser afectados. La vulnerabilidad es el paso 2 (20 de noviembre) y las medidas, el paso 3 (27 de noviembre).

**Requisito.** Los productos de la Práctica 1 en `Mi unidad/CC_Riesgos/practica-1/<zona>/` (sobre todo los de `RD1_02`). Si no están, el cuaderno funciona igual y usted escribe la evidencia de la amenaza a mano.

**Datos.** El cuaderno no descarga nada: lee del repositorio la tabla nacional de exposición por cantón (WorldPop 2020 reescalado al Censo 2022, Copernicus DEM y OpenStreetMap) y los registros de eventos de DesInventar y del ECU 911.

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Complete el formulario de la celda **1** y ejecute las demás en orden. Tiempo estimado: 1 hora en clase y 2 horas sin docente para completar el inventario (sección 9).

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del equipo

In [ ]:
# @title Equipo y provincia (la misma de la Práctica 1)
ESTUDIANTES = "Nombre Apellido · Nombre Apellido"  # @param {type:"string"}
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
ZONA_CAPITULO_1 = ""  # @param {type:"string"}
# Déjelo VACÍO si en la Práctica 1 trabajó a nivel de provincia (lo normal).
# Solo si en la Práctica 1 eligió NIVEL = cantón, escriba aquí ese cantón (por ejemplo «Portoviejo»).
print(f"Equipo: {ESTUDIANTES} · provincia de {PROVINCIA}")

## 2. Entorno de trabajo

In [ ]:
import os, sys, pathlib, subprocess, datetime, json, re, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, riesgo as RG, exposicion as EX

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(os.environ.get("CC_BASE", "."))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, geopandas as gpd
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
SP = limites.slug(PROVINCIA)
_z1 = ZONA_CAPITULO_1.strip().split("·")[-1].strip()
S1 = limites.zona(PROVINCIA, _z1)["slug"] if _z1 else SP   # misma carpeta que creó la Práctica 1
P1 = BASE / "CC_Riesgos" / "practica-1" / S1
RES = BASE / "CC_Riesgos" / "practica-2" / SP
RES.mkdir(parents=True, exist_ok=True)
PROV_FILA, PROV_GEOM = limites.provincia(PROVINCIA)
print(f"Capítulo 1 en {P1} ({'encontrado' if P1.exists() else 'NO encontrado: la evidencia se escribe a mano'})")
print(f"Resultados en {RES}")
META = EX.metadatos()
if META:
    print(f"Tabla de exposición generada el {META.get('generado', '?')[:10]}" + (f" · avisos: {META['errores']}" if META.get("errores") else ""))

## 3. La amenaza: evidencia de su capítulo 1

La exposición siempre es **exposición a algo**. Antes de contar personas y escuelas hay que decir a qué amenazas y con qué evidencia. Esta celda reúne los números del capítulo 1 (`RD1_02`) y el historial de eventos de la provincia.

In [ ]:
EVID = {}
def _leer(nombre, **kw):
    p = P1 / nombre
    return pd.read_csv(p, **kw) if p.exists() else None

RET = _leer(f"RD1_02_periodos_retorno_{S1}.csv")
if RET is not None:
    for (ind, serie), g in RET.groupby(["indice", "serie"]):
        g = g.set_index("periodo_anios")
        for T in (10, 25):
            if T in g.index:
                EVID[f"{ind} {T} años · {serie} (mm)"] = g.loc[T, "nivel"]
IND = _leer(f"RD1_02_indices_anuales_{S1}.csv", header=[0, 1], index_col=0)
if IND is not None:
    f0 = IND.columns.get_level_values(0)[0]
    for k, nom in (("CDD", "Días secos seguidos, media anual"), ("TX90p", "Días cálidos (TX90p), media anual %"),
                   ("R95pTOT", "Lluvia en días muy lluviosos (R95pTOT), media mm")):
        if (f0, k) in IND:
            EVID[f"{nom} · {f0}"] = round(float(IND[(f0, k)].mean()), 1)
TEND = _leer(f"RD1_02_tendencias_{S1}.csv", index_col=[0, 1])
if TEND is not None and "p_valor" in TEND:
    sig = TEND[TEND["p_valor"] < 0.05]
    for (f, k), r in sig.iterrows():
        EVID[f"Tendencia significativa {k} · {f} (por década)"] = round(float(r["sen_por_decada"]), 2)

EV = EX.eventos_por_canton(PROV_FILA["codigo"])
RESUMEN_EV = (EV.groupby(["amenaza", "fuente"])["eventos"].sum().unstack(fill_value=0)
              .rename(index=RG.AMENAZA_CORTA) if len(EV) else pd.DataFrame())
if EVID:
    display(pd.Series(EVID, name="valor").to_frame())
else:
    print("No encontré los productos de RD1_02: escriba la evidencia de cada amenaza en el formulario de la sección 4.")
print("\nEventos registrados en la provincia (DesInventar 1970–2007: desastres; ECU 911 desde 2021: días con incidentes):")
display(RESUMEN_EV)

## 4. Amenazas priorizadas

Marque las amenazas que su capítulo 1 respalda (normalmente dos o tres) y escriba en una línea **el número que lo justifica** (periodo de retorno, factor de El Niño, tendencia, eventos). Las amenazas no marcadas no se califican.

| Amenaza | Qué la respalda en el capítulo 1 |
|---|---|
| Inundación | Rx1day y Rx5day de 10 y 25 años; factor de lluvia en El Niño; eventos de inundación |
| Movimiento en masa | Lluvia acumulada de 5 días; eventos de deslizamiento; relieve |
| Costera | Provincia con costa; eventos de oleaje; El Niño (nivel del mar alto) |
| Sequía | Días secos seguidos (CDD); años de La Niña; eventos de sequía e incendio |
| Calor extremo | Días cálidos (TX90p) y su tendencia |

In [ ]:
# @title Amenazas del territorio
INUNDACION = True  # @param {type:"boolean"}
J_INUNDACION = "Rx1day de 25 años = ___ mm; en El Niño la lluvia de dic–may se multiplica por ___"  # @param {type:"string"}
MOVIMIENTO_MASA = True  # @param {type:"boolean"}
J_MOVIMIENTO_MASA = ""  # @param {type:"string"}
COSTERA = False  # @param {type:"boolean"}
J_COSTERA = ""  # @param {type:"string"}
SEQUIA = False  # @param {type:"boolean"}
J_SEQUIA = ""  # @param {type:"string"}
CALOR = False  # @param {type:"boolean"}
J_CALOR = ""  # @param {type:"string"}

_marcadas = {"inundacion": (INUNDACION, J_INUNDACION), "movimiento_masa": (MOVIMIENTO_MASA, J_MOVIMIENTO_MASA),
             "costera": (COSTERA, J_COSTERA), "sequia": (SEQUIA, J_SEQUIA), "calor": (CALOR, J_CALOR)}
AMENAZAS = [k for k, (v, _) in _marcadas.items() if v]
JUSTIF = {k: j for k, (v, j) in _marcadas.items() if v}
if not AMENAZAS:
    raise ValueError("Marque al menos una amenaza.")
for k in AMENAZAS:
    aviso = "  ⚠ falta la justificación" if not JUSTIF[k].strip() or "___" in JUSTIF[k] else ""
    print(f"· {RG.AMENAZA_CORTA[k]}: {JUSTIF[k]}{aviso}")

## 5. Elementos expuestos por cantón

| Indicador | Qué cuenta | Fuente |
|---|---|---|
| Población 2022 | Habitantes del cantón | WorldPop 2020 (1 km) reescalado al total provincial del Censo 2022 |
| Zona baja | Población, servicios y vías bajo 10 m de altitud | Copernicus DEM GLO-90 |
| Ladera | Población, servicios y vías con pendiente mayor de 20° | Copernicus DEM GLO-90 |
| Servicios críticos | Escuelas y colegios, salud, bomberos y policía | OpenStreetMap |
| Vías principales | Autopistas, troncales, primarias y secundarias (km) y puentes | OpenStreetMap |
| Eventos | Desastres (DesInventar) y días con incidentes (ECU 911) por amenaza | `datos/eventos` |

**Cuidado con las fuentes.** WorldPop distribuye la población con un modelo, no la cuenta casa por casa. OpenStreetMap es colaborativo: en cantones rurales suele faltar información, y un 0 puede significar «no mapeado». La zona baja y la ladera son aproximaciones con un relieve de 90 m: no reemplazan un mapa de amenaza de inundación ni de susceptibilidad a deslizamientos.

In [ ]:
T = EX.de_provincia(PROVINCIA, EV)
COLS = [c for c in ["poblacion_2022", "densidad_hab_km2", "pct_cota_baja", "pct_ladera", "escuelas", "salud", "emergencia",
                    "km_vias_principales", "puentes", "eventos_inundacion", "eventos_movimiento_masa", "eventos_costera",
                    "eventos_sequia"] if c in T]
display(T[["canton"] + COLS].rename(columns=EX.NOMBRES).round(1))
tot = T[[c for c in ("poblacion_2022", "poblacion_cota_baja", "poblacion_ladera", "servicios", "km_vias_principales") if c in T]].sum()
print("Totales de la provincia:", ", ".join(f"{EX.NOMBRES.get(k, k)}: {v:,.0f}" for k, v in tot.items()))

In [ ]:
CANT = limites.cargar("cantones")
G = CANT.merge(T, on="codigo", how="inner")
MAPAS = [(c, EX.NOMBRES[c], cm) for c, cm in (("densidad_hab_km2", "YlOrBr"), ("pct_cota_baja", "Blues"),
                                               ("pct_ladera", "Purples"), ("servicios", "Greens")) if c in G]
fig, axs = plt.subplots(2, 2, figsize=(12, 11)); axs = axs.ravel()
for ax in axs[len(MAPAS):]: ax.set_axis_off()
for ax, (c, titulo, cm) in zip(axs, MAPAS):
    G.plot(column=c, ax=ax, cmap=cm, legend=True, edgecolor="#555555", linewidth=0.4,
           legend_kwds={"shrink": 0.6}, missing_kwds={"color": "#eeeeee"})
    ax.set_title(titulo, fontsize=11); ax.set_axis_off()
fig.suptitle(f"{PROVINCIA}: elementos expuestos por cantón", fontweight="bold"); fig.tight_layout()
fig.savefig(RES / f"RD2_01_mapas_exposicion_{SP}.png", dpi=200); plt.show()

## 6. Calificación de la exposición por amenaza

Para cada amenaza se toman sus indicadores (tabla de abajo), se ordena cada cantón **dentro de la provincia** en quintiles (1 = el 20 % menos expuesto … 5 = el 20 % más expuesto) y se promedian. Un indicador que vale 0 en toda la provincia no se usa.

| Amenaza | Indicadores |
|---|---|
| Inundación | población total y en zona baja, servicios críticos, vías principales, eventos de inundación |
| Movimiento en masa | población, servicios y vías en ladera; eventos de deslizamiento |
| Costera | población, servicios y vías en zona baja; eventos de oleaje |
| Sequía | población, área, eventos de sequía e incendio |
| Calor extremo | población y densidad |

Es una calificación **relativa**: Portoviejo puede tener un 5 en Manabí y no lo tendría comparado con Guayaquil. Para el riesgo del paso 2 se usan estos valores en la misma escala 1–5 de `herramientas/riesgo.py`.

In [ ]:
PUNT, USADOS = EX.calificar(T, AMENAZAS)
for a in AMENAZAS:
    print(f"{RG.AMENAZA_CORTA[a]}: {', '.join(EX.NOMBRES.get(c, c) for c in USADOS[a]) or 'sin indicadores con datos'}")
M = PUNT.set_index("canton")[AMENAZAS].rename(columns=RG.AMENAZA_CORTA)
fig, ax = plt.subplots(figsize=(1.6 * len(AMENAZAS) + 3, 0.34 * len(M) + 1.6))
im = ax.imshow(M.values, cmap="YlOrRd", vmin=1, vmax=5, aspect="auto")
ax.set_xticks(range(M.shape[1]), M.columns, rotation=20, ha="right"); ax.set_yticks(range(len(M)), M.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.values[i, j]
        if np.isfinite(v): ax.text(j, i, int(v), ha="center", va="center", fontsize=9, color="white" if v >= 4 else "black")
plt.colorbar(im, ax=ax, shrink=0.6, label="Exposición (1–5, relativa a la provincia)")
ax.set_title(f"{PROVINCIA}: exposición por cantón y amenaza"); fig.tight_layout()
fig.savefig(RES / f"RD2_01_matriz_exposicion_{SP}.png", dpi=200); plt.show()

## 7. Revisión del equipo

La calificación automática no conoce su territorio. Si sabe que un cantón está más o menos expuesto de lo que dice la matriz (un barrio entero en la ribera, una vía que OpenStreetMap no tiene, un cantón con poca gente en la zona de amenaza), corríjalo y **justifíquelo con su fuente**. Formato: `Cantón: amenaza = valor`, separados por punto y coma. Amenazas: `inundacion`, `movimiento_masa`, `costera`, `sequia`, `calor`.

In [ ]:
# @title Correcciones (opcional)
AJUSTES = ""  # @param {type:"string"}
# Ejemplo: Portoviejo: inundacion = 5; Jipijapa: sequia = 4
J_AJUSTES = ""  # @param {type:"string"}
# Fuente de las correcciones: PDOT del cantón, SitRep de la SNGR, visita, mapa de amenaza del GAD.

FINAL = PUNT.copy()
CAMBIOS = []
for parte in filter(None, (p.strip() for p in AJUSTES.split(";"))):
    m = re.match(r"(.+?):\s*(\w+)\s*=\s*([1-5])$", parte)
    if not m:
        print(f"⚠ No entendí «{parte}»: use el formato «Cantón: amenaza = valor»."); continue
    c, a, v = m.group(1).strip(), m.group(2).strip(), int(m.group(3))
    fila = FINAL["canton"].map(limites.normalizar) == limites.normalizar(c)
    if not fila.any() or a not in AMENAZAS:
        print(f"⚠ «{parte}»: revise el nombre del cantón o la amenaza (marcadas: {', '.join(AMENAZAS)})."); continue
    CAMBIOS.append({"canton": FINAL.loc[fila, "canton"].iloc[0], "amenaza": a, "antes": int(FINAL.loc[fila, a].iloc[0]), "despues": v})
    FINAL.loc[fila, a] = v
if CAMBIOS:
    display(pd.DataFrame(CAMBIOS))
    if not J_AJUSTES.strip(): print("⚠ Falta la fuente de las correcciones (J_AJUSTES).")
else:
    print("Sin correcciones: se usa la calificación automática.")

## 8. Cantones prioritarios

Los tres cantones más expuestos a cada amenaza, con sus números reales. Estos cantones son los que el paso 2 evalúa en vulnerabilidad. En el mapa, la exposición a la primera amenaza marcada.

In [ ]:
PRIOR, TABLAS = [], {}
for a in AMENAZAS:
    cols = [c for c in USADOS[a]] or ["poblacion_2022"]
    sub = FINAL[["codigo", "canton", a]].merge(T[["codigo"] + cols], on="codigo").sort_values([a] + cols[:1], ascending=False).head(3)
    TABLAS[a] = sub.drop(columns="codigo").rename(columns={a: "exposición (1–5)", **EX.NOMBRES}).set_index("canton")
    for _, r in sub.iterrows():
        for c in cols:
            PRIOR.append({"amenaza": RG.AMENAZA_CORTA[a], "clave": a, "canton": r["canton"], "exposicion": int(r[a]),
                          "indicador": EX.NOMBRES.get(c, c), "columna": c, "valor": r[c]})
PRIOR = pd.DataFrame(PRIOR)
for a, tb in TABLAS.items():
    print(f"\n{RG.AMENAZA_CORTA[a]}"); display(tb)
a0 = AMENAZAS[0]
G2 = CANT.merge(FINAL[["codigo", a0]], on="codigo", how="inner")
fig, ax = plt.subplots(figsize=(7, 6))
G2.plot(column=a0, ax=ax, cmap="YlOrRd", vmin=1, vmax=5, legend=True, edgecolor="#555555", linewidth=0.5,
        legend_kwds={"shrink": 0.6, "label": "Exposición 1–5"})
for _, r in G2.iterrows():
    if r[a0] >= 4:
        p = r.geometry.representative_point(); ax.annotate(r["nombre_dpa"].replace("Ð", "Ñ").title(), (p.x, p.y), fontsize=7, ha="center")
ax.set_title(f"{PROVINCIA}: exposición a {RG.AMENAZA_CORTA[a0].lower()}"); ax.set_axis_off(); fig.tight_layout()
fig.savefig(RES / f"RD2_01_mapa_{a0}_{SP}.png", dpi=200); plt.show()

## 9. Inventario de elementos expuestos (trabajo sin docente)

El cuaderno deja una planilla con los cantones prioritarios. Para cada uno, el equipo identifica **elementos concretos** en la zona de amenaza: el barrio en la ribera, la escuela junto a la quebrada, el puente de la vía principal, el sistema de agua, los cultivos. La planilla trae ya los conteos automáticos; usted agrega nombres, ubicación y fuente.

**Fuentes recomendadas:** PDOT del cantón (diagnóstico biofísico y de asentamientos), mapas de amenaza del GAD o de la SNGR, el geoportal del IGM, Google Earth o un recorrido. Cada fila lleva su fuente.

Archivo: `RD2_01_inventario_expuestos_<provincia>.xlsx` en su carpeta de resultados. Ábralo en Google Sheets o Excel, complételo y guárdelo con el mismo nombre: el paso 2 lo lee.

In [ ]:
TIPOS = {"poblacion_cota_baja": "personas", "poblacion_ladera": "personas", "poblacion_2022": "personas",
         "servicios": "servicios", "servicios_cota_baja": "servicios", "servicios_ladera": "servicios",
         "km_vias_principales": "vías", "km_vias_cota_baja": "vías", "km_vias_ladera": "vías"}
INV = []
for _, r in PRIOR.drop_duplicates(["clave", "canton"]).iterrows():
    fila = T[T.canton == r["canton"]].iloc[0]
    for c in USADOS.get(r["clave"], []):
        if c in TIPOS:
            INV.append({"canton": r["canton"], "amenaza": r["amenaza"], "tipo": TIPOS[c], "elemento": EX.NOMBRES[c],
                        "cantidad": fila[c], "nombre_o_lugar": "", "latitud": "", "longitud": "",
                        "fuente": "Tabla de exposición del curso (WorldPop, Copernicus DEM, OSM)", "observacion": ""})
    for _ in range(3):
        INV.append({"canton": r["canton"], "amenaza": r["amenaza"], "tipo": "", "elemento": "", "cantidad": "",
                    "nombre_o_lugar": "", "latitud": "", "longitud": "", "fuente": "", "observacion": ""})
INV = pd.DataFrame(INV)
RUTA_INV = RES / f"RD2_01_inventario_expuestos_{SP}.xlsx"
if RUTA_INV.exists():
    print(f"Ya existe {RUTA_INV.name}: no lo sobrescribo para no perder lo que completó. Bórrelo si quiere uno nuevo.")
else:
    with pd.ExcelWriter(RUTA_INV) as w:
        INV.to_excel(w, sheet_name="inventario", index=False)
        pd.DataFrame({"campo": ["tipo", "elemento", "cantidad", "nombre_o_lugar", "fuente"],
                      "qué escribir": ["personas, viviendas, servicios, vías, economía o ecosistemas",
                                       "qué es (escuela, barrio, puente, captación de agua, cultivo de arroz…)",
                                       "número de personas, viviendas, km o hectáreas",
                                       "nombre propio y sector; coordenadas si las tiene",
                                       "de dónde sale el dato (PDOT 2023 del cantón, SNGR, visita del 15 de noviembre…)"]}
                     ).to_excel(w, sheet_name="instrucciones", index=False)
    print(f"Planilla creada: {RUTA_INV}")
display(INV.head(12))

## 10. Guardar

In [ ]:
T.to_csv(RES / f"RD2_01_tabla_exposicion_{SP}.csv", index=False, encoding="utf-8")
FINAL.to_csv(RES / f"RD2_01_puntajes_exposicion_{SP}.csv", index=False, encoding="utf-8")
PRIOR.to_csv(RES / f"RD2_01_cantones_prioritarios_{SP}.csv", index=False, encoding="utf-8")
if len(EV): EV.to_csv(RES / f"RD2_01_eventos_por_canton_{SP}.csv", index=False, encoding="utf-8")
meta = {"material": "CC-IRD-P2-01 v1.0", "equipo": ESTUDIANTES, "provincia": PROVINCIA, "capitulo_1": str(P1),
        "amenazas": AMENAZAS, "justificacion": JUSTIF, "evidencia_capitulo_1": {k: (None if pd.isna(v) else float(v)) for k, v in EVID.items()},
        "indicadores": USADOS, "correcciones": CAMBIOS, "fuente_correcciones": J_AJUSTES,
        "tabla_exposicion": META.get("generado"), "fecha": datetime.datetime.now().isoformat(timespec="seconds")}
(RES / "RD2_01_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Guardado en", RES)
for p in sorted(RES.glob("RD2_01_*")): print("  ", p.name)

## 11. Preguntas para el capítulo 2

1. ¿Qué amenazas priorizó y con qué número del capítulo 1 lo justifica?
2. ¿Qué tres cantones están más expuestos a su amenaza principal? ¿Qué indicador pesa más en cada uno: la cantidad de gente, la gente en zona baja o ladera, los servicios o el historial de eventos?
3. ¿Hay cantones con mucha población pero poca exposición a esa amenaza (o al revés)? Explique por qué con los indicadores.
4. ¿Coincide la exposición calculada con el historial de eventos? Si un cantón tiene muchos eventos y poca exposición calculada (o al revés), ¿qué podría explicarlo (escala de 90 m y 1 km, OSM incompleto, registros sesgados hacia las ciudades)?
5. ¿Qué corrigió el equipo en la sección 7 y con qué fuente?
6. Del inventario: nombre al menos tres elementos concretos expuestos en cada cantón prioritario (un barrio, un servicio y una vía o un medio de vida).

**Para el informe (limitaciones).** La exposición es relativa a la provincia; la población viene de un modelo (WorldPop) ajustado al Censo; la zona baja (< 10 m) y la ladera (> 20°) son aproximaciones con un relieve de 90 m; OpenStreetMap puede estar incompleto; los eventos del ECU 911 son días con incidentes, no desastres verificados, y hay un vacío de registros entre 2008 y junio de 2021.

**Citas.** IPCC (2022). *Climate Change 2022: Impacts, Adaptation and Vulnerability*, Anexo II: Glosario · WorldPop (2018). *Global High Resolution Population Denominators Project*. University of Southampton. doi:10.5258/SOTON/WP00671 · ESA (2021). *Copernicus DEM GLO-90*. doi:10.5270/ESA-c5d3d65 · OpenStreetMap contributors (2026). Extracto de Ecuador, Geofabrik (ODbL) · INEC (2023). *Censo Ecuador 2022* · McGranahan, G., Balk, D. y Anderson, B. (2007). The rising tide: assessing the risks of climate change and human settlements in low elevation coastal zones. *Environment and Urbanization*, 19(1), 17–37 · UNDRR. *DesInventar Sendai, Ecuador* · SIS ECU 911. *Base de Emergencias*, Datos Abiertos Ecuador.